In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# Load Dataset
url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
df = pd.read_csv(url)

# Define Features
numerical = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']
categorical = ['industry', 'location', 'lead_source', 'employment_status']
features = categorical + numerical

# Split Data (60% Train, 20% Val, 20% Test)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values

df_train_filled = df_train[features].fillna(0)
df_val_filled = df_val[features].fillna(0)

In [2]:
for c in numerical:
    auc = roc_auc_score(y_train, df_train_filled[c])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train_filled[c])
    print(f"Q1 AUC for {c:<25}: {auc:.3f}")

Q1 AUC for interaction_count        : 0.738
Q1 AUC for lead_score               : 0.614
Q1 AUC for number_of_courses_viewed : 0.764
Q1 AUC for annual_income            : 0.552


In [3]:
train_dicts = df_train_filled.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val_filled.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=1)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_val)[:, 1]
auc_val = roc_auc_score(y_val, y_pred_proba)

print(f"Q2 Model AUC: {round(auc_val, 3)}")

Q2 Model AUC: 0.817


In [4]:
thresholds = np.linspace(0, 1, 101)
scores = []

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)
    
    predict_positive = (y_pred_proba >= t)
    predict_negative = (y_pred_proba < t)
    
    tp = (predict_positive & actual_positive).sum()
    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    scores.append((t, p, r))

# Kupata threshold ambapo precision na recall zinintersect
df_scores = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall'])
df_scores['diff'] = np.abs(df_scores['precision'] - df_scores['recall'])
intersect = df_scores.sort_values(by='diff').iloc[0]

print(f"Q3 Intersection Threshold: {intersect['threshold']:.2f} | Precision/Recall Value: {intersect['precision']:.2f}")

Q3 Intersection Threshold: 1.00 | Precision/Recall Value: 0.00


In [5]:
df_scores['f1'] = 2 * (df_scores['precision'] * df_scores['recall']) / (df_scores['precision'] + df_scores['recall'])
max_f1 = df_scores.sort_values(by='f1', ascending=False).iloc[0]

print(f"Q4 Maximal F1 Score Threshold: {max_f1['threshold']:.2f} | Maximal F1 Score: {max_f1['f1']:.2f}")

Q4 Maximal F1 Score Threshold: 0.57 | Maximal F1 Score: 0.81


In [6]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train_fold = df_full_train.iloc[train_idx]
    df_val_fold = df_full_train.iloc[val_idx]
    
    y_train_fold = df_train_fold['converted'].values
    y_val_fold = df_val_fold['converted'].values
    
    train_dicts = df_train_fold[features].fillna(0).to_dict(orient='records')
    X_train_fold = dv.fit_transform(train_dicts)
    
    val_dicts = df_val_fold[features].fillna(0).to_dict(orient='records')
    X_val_fold = dv.transform(val_dicts)
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=1)
    model.fit(X_train_fold, y_train_fold)
    
    y_pred = model.predict_proba(X_val_fold)[:, 1]
    auc = roc_auc_score(y_val_fold, y_pred)
    scores.append(auc)

print(f"Q5 5-Fold CV Standard Deviation: {round(np.std(scores), 3)}")

Q5 5-Fold CV Standard Deviation: 0.036


In [7]:
for C in [0.000001, 0.001, 1]:
    scores = []
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_fold = df_full_train.iloc[train_idx]
        df_val_fold = df_full_train.iloc[val_idx]
        
        y_train_fold = df_train_fold['converted'].values
        y_val_fold = df_val_fold['converted'].values
        
        train_dicts = df_train_fold[features].fillna(0).to_dict(orient='records')
        X_train_fold = dv.fit_transform(train_dicts)
        
        val_dicts = df_val_fold[features].fillna(0).to_dict(orient='records')
        X_val_fold = dv.transform(val_dicts)
        
        model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=1)
        model.fit(X_train_fold, y_train_fold)
        
        y_pred = model.predict_proba(X_val_fold)[:, 1]
        auc = roc_auc_score(y_val_fold, y_pred)
        scores.append(auc)
        
    print(f"C={C:<10} | Mean AUC: {np.mean(scores):.3f} | Std: {np.std(scores):.3f}")

C=1e-06      | Mean AUC: 0.560 | Std: 0.024
C=0.001      | Mean AUC: 0.867 | Std: 0.029
C=1          | Mean AUC: 0.822 | Std: 0.036
